# Model Evaluation

## Objective

This notebook evaluates the machine learning models implemented from scratch for the Crop Recommendation System.

The following models are evaluated:

- Logistic Regression
- Decision Tree
- Random Forest

The evaluation includes:

- Accuracy
- Confusion Matrix
- Precision
- Recall
- F1-Score
- Model Comparison

Finally, the best-performing model is identified for deployment.

In [1]:
# ==========================================================
# Import Libraries
# ==========================================================

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [2]:
# ==========================================================
# Load Test Dataset
# ==========================================================

X_test = pd.read_csv("../data/processed/X_test_scaled.csv").values
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Testing Features :", X_test.shape)
print("Testing Labels   :", y_test.shape)

Testing Features : (440, 7)
Testing Labels   : (440,)


In [3]:
# ==========================================================
# Number of Classes
# ==========================================================

num_classes = len(np.unique(y_test))

print("Number of Classes:", num_classes)

Number of Classes: 22


In [4]:
# ==========================================================
# Load Logistic Regression Model
# ==========================================================

with open("../saved_models/logistic_regression.pkl", "rb") as file:
    logistic_model = pickle.load(file)

weights = logistic_model["weights"]
bias = logistic_model["bias"]

print("Logistic Regression Model Loaded Successfully")

Logistic Regression Model Loaded Successfully


In [5]:
# ==========================================================
# Load Random Forest Model
# ==========================================================

with open("../saved_models/random_forest_model.pkl", "rb") as file:
    random_forest = pickle.load(file)

print("Random Forest Model Loaded Successfully")

Random Forest Model Loaded Successfully


In [6]:
# ==========================================================
# Evaluation Functions (From Scratch)
# ==========================================================

def calculate_accuracy(y_true, y_pred):
    """
    Calculate classification accuracy.
    """
    correct = np.sum(y_true == y_pred)
    return correct / len(y_true)


def confusion_matrix_scratch(y_true, y_pred, num_classes):
    """
    Generate confusion matrix.
    """
    matrix = np.zeros((num_classes, num_classes), dtype=int)

    for actual, predicted in zip(y_true, y_pred):
        matrix[actual][predicted] += 1

    return matrix


def precision_recall_f1(conf_matrix):
    """
    Calculate Precision, Recall and F1-Score
    for each class and return macro averages.
    """

    num_classes = conf_matrix.shape[0]

    precision_list = []
    recall_list = []
    f1_list = []

    for i in range(num_classes):

        tp = conf_matrix[i, i]

        fp = np.sum(conf_matrix[:, i]) - tp

        fn = np.sum(conf_matrix[i, :]) - tp

        # Precision
        if tp + fp == 0:
            precision = 0
        else:
            precision = tp / (tp + fp)

        # Recall
        if tp + fn == 0:
            recall = 0
        else:
            recall = tp / (tp + fn)

        # F1 Score
        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    return (
        np.mean(precision_list),
        np.mean(recall_list),
        np.mean(f1_list)
    )

In [7]:
# ==========================================================
# Plot Confusion Matrix
# ==========================================================

def plot_confusion_matrix(conf_matrix, title):

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt="d",
        cmap="Blues"
    )

    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")

    plt.tight_layout()
    plt.show()